0. More EDAs on Olympic data
In lecture 03 we ingested Olympic data from Kaggle and did very simple EDA. Now create more EDAs there.

  a) Start with reading in the dataset into a dataframe using spark.

  b) Use spark columns method to find out the columns

  c) Find out the 10 oldest atheletes, their age and the sport

  d) Find out the 10 youngest atheletes, their age and the sport

  e) Find out the five sports with highest median age

  f) Find out the five sports with lowest median age

  g) Find out top 10 countries after number of gold medals

  h) Find out top 10 countries after number of medals

  i) Plot a time series line chart of number of female and male atheletes in same graph.

  j) Do more explorations on your own



In [0]:
# a) Start with reading in the dataset into a dataframe using spark.

DATA_PATH = "/Volumes/data/olympic_games/raw_data"

df_olympic_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True)
df_olympic_athletes.display()

In [0]:
# b) Use spark columns method to find out the columns

df_olympic_athletes.printSchema()

In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])



df_olympic_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_olympic_athletes_schema)

In [0]:
df_olympic_athletes_schema.createOrReplaceTempView("df_olympic_athletes_schema")

spark.sql("""
          SELECT
            DISTINCT sport
          FROM df_olympic_athletes_schema
          
          """).display()

In [0]:
# c) Find out the 10 oldest atheletes, their age and the sport

spark.sql("""
          SELECT
            Name,
            Age,
            Sport
            FROM df_olympic_athletes_schema
            WHERE Age IS NOT NULL
            ORDER BY Age DESC
            LIMIT 10
          """).display()

In [0]:
# d) Find out the 10 youngest atheletes, their age and the sport


spark.sql("""
          SELECT
            name,
            Age,
            sport
          FROM df_olympic_athletes_schema
          WHERE Age IS NOT NULL
          ORDER BY Age ASC
          LIMIT 10
          """).display()

In [0]:
# e) Find out the five sports with highest median age

spark.sql("""
          SELECT
            Sport,
            percentile_approx(Age, 0.5) as median_age
          FROM df_olympic_athletes_schema
          WHERE Age IS NOT NULL
          GROUP BY Sport
          ORDER BY median_age DESC
          LIMIT 5
          """).display()

In [0]:
# f) Find out the five sports with lowest median age

spark.sql("""
          SELECT
            Sport,
            percentile_approx(Age, 0.5) as median_age
          FROM df_olympic_athletes_schema
          WHERE Age IS NOT NULL
          GROUP BY Sport
          ORDER BY median_age ASC
          LIMIT 5
          """).display()

In [0]:
# g) Find out top 10 countries after number of gold medals

spark.sql("""
          SELECT
          NOC,
          count(Medal) as gold_medal_count
          FROM df_olympic_athletes_schema
          WHERE Medal = 'Gold'
          GROUP BY NOC
          ORDER BY gold_medal_count DESC
          LIMIT 10
          """).display()

In [0]:
# h) Find out top 10 countries after number of medals

spark.sql("""
          SELECT
          NOC,
          count(Medal) as medal_count
          FROM df_olympic_athletes_schema
          WHERE Medal IS NOT NULL
          GROUP BY NOC
          ORDER BY medal_count DESC
          LIMIT 10
          """).display()

In [0]:
# i) Plot a time series line chart of number of female and male atheletes in same graph.

df_gender = spark.sql("""
          SELECT
            Year,
            count(case when Sex = 'F' then 1 end) as female_count,
            count(case when Sex = 'M' then 1 end) as male_count
          FROM df_olympic_athletes_schema
          WHERE Year IS NOT NULL
          GROUP BY Year
          """)
df_gender.display()

df_gender.orderBy("Year").toPandas().plot(
    x="Year",
    y=["female_count", "male_count"],
    kind="line",
    figsize=(12, 5),
    title="Olympic athletes by gender"
)

In [0]:
# j) Do more explorations on your own

spark.sql("""
          SELECT
            Sport,
            count(Medal) as gold_medal_count
          FROM df_olympic_athletes_schema
          WHERE Medal = 'Gold'
          GROUP BY Sport
          ORDER BY gold_medal_count DESC
          LIMIT 10
          """).display()